In [ ]:
import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from typing import TypedDict, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langchain_experimental.utilities import PythonREPL

In [ ]:
# Replace with your actual GCP project ID
PROJECT_ID = "your-gcp-project-id" 
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Initialize Gemini Flash
llm = ChatVertexAI(
    model_name="gemini-1.5-flash-001",
    temperature=0.0,
    max_output_tokens=2048,
)

In [ ]:
# 1. State Definition
class CodeGraphState(TypedDict):
    task_description: str
    code: Optional[str]
    test_cases: Optional[str]
    execution_errors: Optional[str]
    iterations: int

# 2. Pydantic Schemas for Strict Output
class CodeOutput(BaseModel):
    python_code: str = Field(description="Raw, executable Python code solving the task. No markdown blocks.")

class TestOutput(BaseModel):
    python_tests: str = Field(description="Raw `assert` statements testing the code. No markdown blocks.")

In [ ]:
def generate_code_node(state: CodeGraphState):
    print(f"--- GENERATING CODE (Iteration {state.get('iterations', 0) + 1}) ---")
    task = state["task_description"]
    errors = state.get("execution_errors")
    
    system_instruction = (
        "You are an expert Python software engineer. "
        "Output ONLY raw, executable code based on the schema. "
        "Do not include explanations or markdown formatting like ```python."
    )
    human_instruction = f"Write Python code for the following task: {task}"
    
    if errors:
        human_instruction += f"\n\nYour previous code failed with these errors. Fix them:\n{errors}"
        
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_instruction),
        ("human", human_instruction)
    ])
    
    structured_llm = llm.with_structured_output(CodeOutput)
    chain = prompt | structured_llm
    response = chain.invoke({})
    
    return {
        "code": response.python_code,
        "iterations": state.get("iterations", 0) + 1,
        "execution_errors": None 
    }

def generate_tests_node(state: CodeGraphState):
    print("--- GENERATING TESTS ---")
    task = state["task_description"]
    code = state["code"]
    
    system_instruction = "You are a QA engineer. Output ONLY raw Python assert statements based on the schema."
    human_instruction = f"Given this task: {task}\nAnd this code:\n{code}\nGenerate edge-case Python test cases using `assert` statements."
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_instruction),
        ("human", human_instruction)
    ])
    
    structured_llm = llm.with_structured_output(TestOutput)
    chain = prompt | structured_llm
    response = chain.invoke({})
    
    return {"test_cases": response.python_tests}

def execute_code_node(state: CodeGraphState):
    print("--- EXECUTING CODE & TESTS ---")
    repl = PythonREPL()
    code_to_run = f"{state['code']}\n\n{state['test_cases']}"
    
    try:
        result = repl.run(code_to_run)
        if "Traceback" in result or "AssertionError" in result:
            print("Execution Failed!")
            return {"execution_errors": result}
        else:
            print("Execution Passed!")
            return {"execution_errors": "SUCCESS"}
    except Exception as e:
        print(f"System Exception: {e}")
        return {"execution_errors": str(e)}

In [ ]:
def check_success(state: CodeGraphState):
    if state["execution_errors"] == "SUCCESS":
        return "success"
    if state["iterations"] >= 3:
        return "max_iterations"
    return "failure"

# Initialize graph
builder = StateGraph(CodeGraphState)

# Add nodes
builder.add_node("coder", generate_code_node)
builder.add_node("tester", generate_tests_node)
builder.add_node("executor", execute_code_node)

# Set edges
builder.set_entry_point("coder")
builder.add_edge("coder", "tester")
builder.add_edge("tester", "executor")

# Add conditional routing
builder.add_conditional_edges(
    "executor",
    check_success,
    {
        "success": END,
        "max_iterations": END,
        "failure": "coder" # Route back to fix code
    }
)

coding_agent = builder.compile()

In [ ]:
task = "Write a Python function to calculate the Levenshtein distance between two strings."

initial_state = {
    "task_description": task,
    "iterations": 0
}

# Run the graph and stream the state updates
for step in coding_agent.stream(initial_state):
    for node, state in step.items():
        pass # The print statements inside the nodes will log the progress

# Print final outputs
final_state = step[list(step.keys())[0]]

print("\n\n===== FINAL GENERATED CODE =====")
print(final_state["code"])

if final_state["execution_errors"] != "SUCCESS":
    print("\nWARNING: Max iterations reached before tests passed.")

In [ ]:
def coder_node(state: AgentState):
    print(f"\n--- CODER NODE (Iteration {state.get('iterations', 0) + 1}) ---")
    
    system_prompt = "You are an expert Python coder. Write clean, PEP8 compliant code to solve the user's task. ONLY output Python code inside ```python ``` blocks. Do not explain the code. Do not write test cases."
    user_prompt = f"Task: {state['task']}\n"
    
    if state.get("feedback") and state.get("error"):
        user_prompt += f"\nPrevious Error:\n{state['error']}\n\nDebugger Advice:\n{state['feedback']}\n\nPlease fix the code."
        
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    response = coder_llm.invoke(messages)
    
    print("Qwen generated new code.")
    return {"code": extract_code(response.content), "iterations": state.get("iterations", 0) + 1}


def test_generator_node(state: AgentState):
    print("\n--- TEST GENERATOR NODE (Phi-3.5) ---")
    
    system_prompt = (
        "You are a strict QA engineer. Write a comprehensive pytest suite for the provided Python code. "
        "The code you are testing will be saved in a file named `solution.py`. "
        "You MUST start your script with `import pytest` and `from solution import *`. "
        "Write edge cases and standard cases. ONLY output the Python test code inside ```python ``` blocks."
    )
    
    user_prompt = f"Original Task: {state['task']}\n\nCode to test:\n```python\n{state['code']}\n```"
    
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    response = tester_llm.invoke(messages)
    
    print("Phi-3.5 generated pytest script.")
    return {"test_code": extract_code(response.content)}


def executor_node(state: AgentState):
    print("\n--- EXECUTOR NODE (Sandbox) ---")
    
    sandbox_dir = "sandbox"
    os.makedirs(sandbox_dir, exist_ok=True)
    
    # Save the main code
    solution_path = os.path.join(sandbox_dir, "solution.py")
    with open(solution_path, "w") as f:
        f.write(state["code"])
        
    # Save the test code
    test_path = os.path.join(sandbox_dir, "test_solution.py")
    with open(test_path, "w") as f:
        f.write(state["test_code"])
        
    try:
        # Run pytest on the generated test file
        result = subprocess.run(
            ["pytest", test_path, "-v"], 
            capture_output=True, 
            text=True, 
            timeout=15
        )
        
        if result.returncode == 0:
            print("Tests Passed Successfully!")
            return {"error": "None", "feedback": ""}
        else:
            print("Tests Failed.")
            return {"error": result.stdout + "\n" + result.stderr} # Pytest puts most info in stdout
            
    except subprocess.TimeoutExpired:
        print("⏳ Execution timed out!")
        return {"error": "TimeoutExpired: The tests took too long to run."}


def debugger_node(state: AgentState):
    print("\n--- DEBUGGER NODE ---")
    
    system_prompt = "You are an expert Python debugger. Analyze the code, the test cases, and the pytest error traceback. Explain exactly WHY it failed and provide a brief, actionable plan to fix the main code. Do NOT write the final code."
    
    user_prompt = (
        f"Main Code:\n```python\n{state['code']}\n```\n\n"
        f"Test Code:\n```python\n{state['test_code']}\n```\n\n"
        f"Pytest Traceback:\n{state['error']}"
    )
    
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    response = debugger_llm.invoke(messages)
    
    print("Phi-3.5 generated debugging plan.")
    return {"feedback": response.content}

## Testing the nodes

In [ ]:
# Create a mock state for the coder
mock_initial_state = {
    "task": "Write a Python function called `is_palindrome` that checks if a given string is a palindrome. It should ignore spaces and casing.",
    "code": "",
    "test_code": "",
    "error": "",
    "feedback": "",
    "iterations": 0
}

# Run the node directly
print("Testing Coder Node...\n")
coder_output = coder_node(mock_initial_state)

# Print the exact string that was extracted
print("\n--- EXTRACTED CODE ---")
print(coder_output.get("code", "No code returned."))
print("\n--- UPDATED ITERATION COUNT ---")
print(coder_output.get("iterations"))

In [ ]:
# Create a mock state containing the code we want to test
mock_testing_state = {
    "task": "Write a Python function called `is_palindrome` that checks if a given string is a palindrome. It should ignore spaces and casing.",
    "code": coder_output.get("code", "def is_palindrome(s):\n    s = s.replace(' ', '').lower()\n    return s == s[::-1]"), # Fallback code just in case
    "test_code": "",
    "error": "",
    "feedback": "",
    "iterations": 1
}

# Run the test generator node directly
print("Testing Test Generator Node...\n")
tester_output = test_generator_node(mock_testing_state)

# Print the exact string that Phi generated
print("\n--- EXTRACTED PYTEST CODE ---")
print(tester_output.get("test_code", "No test code returned."))

## Graph Orchestration & Routing Logic

In [ ]:
def route_after_test(state: AgentState):
    if state["error"] == "None":
        return "end"
    elif state["iterations"] >= 3:
        print("\nMax iterations reached. Aborting.")
        return "end"
    else:
        return "debug"

workflow = StateGraph(AgentState)

# Add all 4 Nodes
workflow.add_node("Coder", coder_node)
workflow.add_node("TestGenerator", test_generator_node)
workflow.add_node("Executor", executor_node)
workflow.add_node("Debugger", debugger_node)

# Set the new cyclical edges
workflow.set_entry_point("Coder")
workflow.add_edge("Coder", "TestGenerator")
workflow.add_edge("TestGenerator", "Executor")

workflow.add_conditional_edges(
    "Executor",
    route_after_test,
    {
        "end": END,
        "debug": "Debugger"
    }
)
workflow.add_edge("Debugger", "Coder")

memory = MemorySaver()
agent = workflow.compile(checkpointer=memory)

In [ ]:
# def route_after_test(state: AgentState):
#     if state["error"] == "None":
#         return "end"
#     elif state["iterations"] >= 3: # Max 3 attempts
#         print("\nMax iterations reached. Aborting.")
#         return "end"
#     else:
#         return "debug"

# # Build the Graph
# workflow = StateGraph(AgentState)

# # Add Nodes
# workflow.add_node("Coder", coder_node)
# workflow.add_node("Tester", tester_node)
# workflow.add_node("Debugger", debugger_node)

# # Add Edges
# workflow.set_entry_point("Coder")
# workflow.add_edge("Coder", "Tester")
# workflow.add_conditional_edges(
#     "Tester",
#     route_after_test,
#     {
#         "end": END,
#         "debug": "Debugger"
#     }
# )
# workflow.add_edge("Debugger", "Coder")

# # Initialize memory and compile
# memory = MemorySaver()
# agent = workflow.compile(checkpointer=memory)

## Running the Agent

In [ ]:
# The task
initial_state = {
    # "task": "Write a python script that creates a list of numbers from 1 to 10, calculates the square root of each, and prints the result. Ensure you import the math module.",
    "task": "Solve using python: You are climbing a staircase. It takes`n`steps to reach the top.Each time you can either climb`1`or`2`steps. In how many distinct ways can you climb to the top?. Input: n = 3",
    "code": "",
    "error": "",
    "feedback": "",
    "iterations": 0
}


# Create a config for the checkpointer
config = {"configurable": {"thread_id": "loyums_first_agent"}}

# Stream the execution, passing the config
for output in agent.stream(initial_state, config=config):
    print(list(output.keys())) # Shows which node just ran

print("\n\n--- FINAL DELIVERED CODE ---")
# Now get_state will work perfectly!
final_state = agent.get_state(config).values
print(final_state.get('code', 'No code generated.'))


--- CODER NODE (Iteration 1) ---
Qwen generated new code.
['Coder']

--- TESTER NODE ---
Execution Successful!
['Tester']


--- FINAL DELIVERED CODE ---
def climb_stairs(n):
    if n == 1:
        return 1
    a, b = 1, 2
    for i in range(3, n + 1):
        temp = b
        b = a + b
        a = temp
    return b

n = 3
print(climb_stairs(n))
